# Σύστημα Β – Βήμα 4: συνάρτηση πρόβλεψης για νέες συνεδρίες
Το αρχείο `src/predict_intrusion.py` κάνει για το Σύστημα Β ό,τι το `predict_unsw.py` για το Α: δίνεις συνεδρίες, παίρνεις
**πιθανότητα επίθεσης και απόφαση του νευρωνικού**, και δίπλα **την απόφαση του απλού κανόνα με την αιτία της**.

Έτσι η συνάρτηση δείχνει και το κεντρικό εύρημα του Β: στις περισσότερες συνεδρίες δίκτυο και κανόνας λένε το ίδιο, και ο κανόνας εξηγεί το γιατί.

1. Εξαγωγή του κανόνα από το δέντρο απόφασης και αποθήκευση.
2. Έλεγχος: ίδια αποτελέσματα με το B03.
3. Πώς φαίνεται η έξοδος, μία συνεδρία, λάθος δεδομένα, χρήση από το τερματικό.

Χρόνος: κάτω από 1 λεπτό.

In [1]:
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score

DATA, MODELS = "../data/", "../models/"
SEED = 42

with open(MODELS + "intrusion_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
d = np.load(DATA + "intrusion_prepared.npz")
preprocessor = joblib.load(MODELS + "intrusion_preprocessor.joblib")
feature_names = meta["feature_names"]

# οι αρχικές γραμμές (σε πραγματικές μονάδες), όπως στο B01
df = pd.read_csv(DATA + "intrusion_clean.csv", keep_default_na=False)
df["browser_type"] = pd.read_csv(DATA + "cybersecurity_intrusion_data.csv")["browser_type"].to_numpy()
raw = {s: df.loc[d[f"idx_{s}"]] for s in ("train", "val", "test")}
y = {s: d[f"y_{s}"].astype(int) for s in ("train", "val", "test")}
assert all((raw[s][meta["target"]].to_numpy() == y[s]).all() for s in raw), "Οι γραμμές δεν ταιριάζουν με το B01"
print({s: len(r) for s, r in raw.items()})

{'train': 6675, 'val': 1431, 'test': 1431}


## Β4.1 Ο κανόνας του δέντρου, σε μορφή που διαβάζει η συνάρτηση
Το δέντρο βάθους 3 (ίδιο με B02/B03) μετατρέπεται σε λίστα από όρους στις **πραγματικές μονάδες**. Κάθε διαδρομή που καταλήγει σε «επίθεση» γίνεται ένας κανόνας,
και ο συνολικός κανόνας είναι «επίθεση αν ισχύει έστω μία διαδρομή». Περιττοί όροι αφαιρούνται (π.χ. «failed_logins ≤ 2.5 και login_attempts > 6.5» γίνεται «login_attempts > 6.5»,
αφού το «failed_logins > 2.5» είναι ήδη κανόνας από μόνο του).

Στο τέλος ελέγχουμε ότι ο απλοποιημένος κανόνας δίνει **ακριβώς** τις ίδιες αποφάσεις με το δέντρο.

In [2]:
tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(d["X_train"], y["train"])

def condition(fname, thr, go_right):
    prefix, col = fname.split("__", 1)
    if prefix == "cat":
        for c in meta["cat_cols"]:
            if col.startswith(c + "_"):
                return {"feature": c, "op": "==" if go_right else "!=", "value": col[len(c) + 1:]}
    if prefix == "num":
        sc = preprocessor.named_transformers_["num"]; i = list(sc.feature_names_in_).index(col)
        value = thr * sc.scale_[i] + sc.mean_[i]
    elif prefix == "log":
        sc = preprocessor.named_transformers_["log"].named_steps["scale"]; i = list(sc.feature_names_in_).index(col)
        value = np.expm1(thr * sc.scale_[i] + sc.mean_[i])
    else:
        value = thr
    return {"feature": col, "op": ">" if go_right else "<=", "value": float(value)}

def attack_paths(node=0, path=()):
    t = tree.tree_
    if t.children_left[node] == -1:
        return [list(path)] if t.value[node][0].argmax() == 1 else []
    f, thr = feature_names[t.feature[node]], t.threshold[node]
    return (attack_paths(t.children_left[node], path + (condition(f, thr, False),)) +
            attack_paths(t.children_right[node], path + (condition(f, thr, True),)))

NEG = {">": "<=", "<=": ">", "==": "!=", "!=": "=="}
def simplify(paths):
    paths = sorted(paths, key=len)
    changed = True
    while changed:
        changed = False
        singles = [p[0] for p in paths if len(p) == 1]
        for p in paths:
            for c in list(p):
                if len(p) > 1 and any(s["feature"] == c["feature"] and s["value"] == c["value"] and s["op"] == NEG[c["op"]] for s in singles):
                    p.remove(c); changed = True
        paths = sorted(paths, key=len)
    return paths

paths = simplify(attack_paths())
for i, p in enumerate(paths, 1):
    print(f"Κανόνας {i}: επίθεση αν " + " και ".join(f"{c['feature']} {c['op']} {c['value']:.4g}" if isinstance(c['value'], float) else f"{c['feature']} {c['op']} {c['value']}" for c in p))

def apply_rule(X):
    ops = {">": np.greater, "<=": np.less_equal, "==": np.equal, "!=": np.not_equal}
    fired = np.zeros(len(X), bool)
    for p in paths:
        hit = np.ones(len(X), bool)
        for c in p:
            hit &= ops[c["op"]](X[c["feature"]].to_numpy(), c["value"])
        fired |= hit
    return fired.astype(int)

same = {s: (apply_rule(raw[s]) == tree.predict(d[f"X_{s}"])).mean() for s in raw}
print("\nΊδιες αποφάσεις με το δέντρο:", {s: f"{v:.2%}" for s, v in same.items()})
assert all(v == 1 for v in same.values()), "Ο κανόνας δεν ταυτίζεται με το δέντρο – πες το μου"
with open(MODELS + "intrusion_rule.json", "w", encoding="utf-8") as f:
    json.dump({"source": "δέντρο απόφασης βάθους 3 (notebook B04), όροι σε πραγματικές μονάδες", "attack_paths": paths},
              f, ensure_ascii=False, indent=2)
print("Αποθηκεύτηκε: models/intrusion_rule.json")

Κανόνας 1: επίθεση αν failed_logins > 2.5
Κανόνας 2: επίθεση αν login_attempts > 6.5
Κανόνας 3: επίθεση αν ip_reputation_score > 0.5999

Ίδιες αποφάσεις με το δέντρο: {'train': '100.00%', 'val': '100.00%', 'test': '100.00%'}
Αποθηκεύτηκε: models/intrusion_rule.json


## Β4.2 Έλεγχος: ίδια αποτελέσματα με το B03
Δίνουμε στη συνάρτηση τις **αρχικές** γραμμές του test (σε πραγματικές μονάδες, με κείμενο στις κατηγορικές στήλες), όπως θα της έδινε κάποιος νέα δεδομένα.

In [3]:
sys.path.append("../src")
from predict_intrusion import IntrusionDetector

detector = IntrusionDetector(MODELS)
print("Στήλες εισόδου:", detector.features, "| κατώφλι:", detector.threshold)

pred = detector.predict(raw["test"])
yt = y["test"]
fpr = lambda p: ((p == 1) & (yt == 0)).sum() / (yt == 0).sum()
checks = [("νευρωνικό: accuracy", accuracy_score(yt, pred["is_attack"]), 0.8826),
          ("νευρωνικό: recall", recall_score(yt, pred["is_attack"]), 0.7422),
          ("νευρωνικό: ψευδείς συναγερμοί (FPR)", fpr(pred["is_attack"]), 0.0038),
          ("κανόνας: accuracy", accuracy_score(yt, pred["rule_attack"]), 0.8630),
          ("κανόνας: ψευδείς συναγερμοί (FPR)", fpr(pred["rule_attack"]), 0.0000)]
table = pd.DataFrame(checks, columns=["μέτρο", "συνάρτηση", "αναμενόμενο (B03)"]).set_index("μέτρο")
table["ίδιο"] = np.where((table["συνάρτηση"] - table["αναμενόμενο (B03)"]).abs() < 0.0005, "ΝΑΙ", "ΟΧΙ")
print(table.round(4).to_string())
print(f"\nΊδια απόφαση δικτύου και κανόνα στο test: {(pred['is_attack'] == pred['rule_attack']).mean():.1%}")

Στήλες εισόδου: ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access', 'protocol_type', 'encryption_used', 'browser_type'] | κατώφλι: 0.5
                                     συνάρτηση  αναμενόμενο (B03) ίδιο
μέτρο                                                                 
νευρωνικό: accuracy                     0.8826             0.8826  ΝΑΙ
νευρωνικό: recall                       0.7422             0.7422  ΝΑΙ
νευρωνικό: ψευδείς συναγερμοί (FPR)     0.0038             0.0038  ΝΑΙ
κανόνας: accuracy                       0.8630             0.8630  ΝΑΙ
κανόνας: ψευδείς συναγερμοί (FPR)       0.0000             0.0000  ΝΑΙ

Ίδια απόφαση δικτύου και κανόνα στο test: 97.5%


## Β4.3 Πώς φαίνεται η έξοδος
Δέκα τυχαίες συνεδρίες του test, με την πραγματική ετικέτα αριστερά.

In [4]:
sample = raw["test"].sample(10, random_state=7).index
view = pd.concat([raw["test"].loc[sample, [meta["target"]]].rename(columns={meta["target"]: "πραγματικό"}),
                  raw["test"].loc[sample, detector.features], pred.loc[sample]], axis=1)
pd.set_option("display.max_colwidth", 80)
print(view.round(3).to_string())

      πραγματικό  network_packet_size  login_attempts  session_duration  ip_reputation_score  failed_logins  unusual_time_access protocol_type encryption_used browser_type  p_attack  is_attack  rule_attack                   rule_reason
1121           0                  737               2          2495.413                0.152              2                    0           TCP             DES         Edge     0.196          0            0                              
4819           0                  444               5           110.870                0.432              2                    0           TCP             AES      Firefox     0.200          0            0                              
618            0                  596               1            15.540                0.459              1                    0           UDP             AES       Safari     0.140          0            0                              
7124           0                  768               5   

## Β4.4 Μία μόνο συνεδρία
Η συνάρτηση δέχεται και ένα απλό `dict` με τις 9 τιμές. Οι κατηγορικές δίνονται ως κείμενο, όπως στο αρχικό αρχείο.

In [5]:
session = {"network_packet_size": 520, "protocol_type": "TCP", "login_attempts": 8, "session_duration": 410.5,
           "encryption_used": "AES", "ip_reputation_score": 0.21, "failed_logins": 1, "browser_type": "Chrome",
           "unusual_time_access": 0}
print(detector.predict(session).round(3).T.to_string(header=False))

p_attack                      1.0
is_attack                       1
rule_attack                     1
rule_reason  login_attempts > 6.5


## Β4.5 Τι γίνεται με λάθος δεδομένα

In [6]:
good = raw["test"].loc[sample[:3], detector.features]
cases = {"λείπει στήλη": good.drop(columns=["failed_logins"]),
         "κενή τιμή": good.assign(session_duration=[np.nan, 1.0, 2.0]),
         "κείμενο αντί για αριθμό": good.astype(object).assign(login_attempts=["πολλές", 2, 3]),
         "αρνητική τιμή": good.assign(ip_reputation_score=[-0.1, 0.2, 0.3])}
for name, df_bad in cases.items():
    try:
        detector.predict(df_bad)
        print(f"{name}: ΔΕΝ εντοπίστηκε πρόβλημα (δεν θα έπρεπε)")
    except (ValueError, TypeError) as e:
        print(f"{name}: {e}")
print("Άγνωστος browser (δεκτός):", detector.predict(good.assign(browser_type="Opera"))["is_attack"].tolist())

λείπει στήλη: Λείπουν στήλες εισόδου: ['failed_logins']. Απαιτούνται: ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access', 'protocol_type', 'encryption_used', 'browser_type']
κενή τιμή: Κενές ή μη αριθμητικές τιμές στις στήλες: ['session_duration']
κείμενο αντί για αριθμό: Κενές ή μη αριθμητικές τιμές στις στήλες: ['login_attempts']
αρνητική τιμή: Αρνητικές τιμές στις στήλες: ['ip_reputation_score'] (όλες οι αριθμητικές είσοδοι είναι μη αρνητικές).
Άγνωστος browser (δεκτός): [0, 0, 0]


## Β4.6 Από αρχείο CSV
Φτιάχνουμε ένα μικρό αρχείο με 20 συνεδρίες (μόνο οι 9 στήλες εισόδου, χωρίς την ετικέτα) και το δίνουμε στη συνάρτηση από το τερματικό.

In [7]:
raw["test"].sample(20, random_state=3)[detector.features].to_csv(DATA + "example_sessions.csv", index=False)
print("Αποθηκεύτηκε: data/example_sessions.csv")

Αποθηκεύτηκε: data/example_sessions.csv


In [8]:
base = {"network_packet_size": 500, "protocol_type": "TCP", "login_attempts": 3, "session_duration": 600,
        "encryption_used": "AES", "ip_reputation_score": 0.2, "failed_logins": 1,
        "browser_type": "Chrome", "unusual_time_access": 0}

def what_if(col, values):
    rows = pd.DataFrame([{**base, col: v} for v in values])
    return pd.concat([rows[[col]], detector.predict(rows).round(3)], axis=1)

for col, values in [("failed_logins", [0, 1, 2, 3, 4, 5]),
                    ("login_attempts", [1, 3, 5, 6, 7, 9, 12]),
                    ("ip_reputation_score", [0.1, 0.3, 0.5, 0.59, 0.61, 0.8]),
                    ("browser_type", ["Chrome", "Firefox", "Edge", "Safari", "Unknown"]),
                    ("network_packet_size", [100, 500, 1200])]:
    print(f"\n--- αλλάζει μόνο το {col} ---")
    print(what_if(col, values).to_string(index=False))


--- αλλάζει μόνο το failed_logins ---
 failed_logins  p_attack  is_attack  rule_attack         rule_reason
             0     0.107          0            0                    
             1     0.109          0            0                    
             2     0.126          0            0                    
             3     0.998          1            1 failed_logins > 2.5
             4     1.000          1            1 failed_logins > 2.5
             5     1.000          1            1 failed_logins > 2.5

--- αλλάζει μόνο το login_attempts ---
 login_attempts  p_attack  is_attack  rule_attack          rule_reason
              1     0.097          0            0                     
              3     0.109          0            0                     
              5     0.121          0            0                     
              6     0.138          0            0                     
              7     0.976          1            1 login_attempts > 6.5
            

In [9]:
rows = pd.DataFrame([{**base, "browser_type": b, "login_attempts": n}
                     for b in ["Chrome", "Unknown"] for n in [3, 4, 5, 6]])
print(pd.concat([rows[["browser_type", "login_attempts"]], detector.predict(rows).round(3)], axis=1).to_string(index=False))

browser_type  login_attempts  p_attack  is_attack  rule_attack rule_reason
      Chrome               3     0.109          0            0            
      Chrome               4     0.116          0            0            
      Chrome               5     0.121          0            0            
      Chrome               6     0.138          0            0            
     Unknown               3     0.163          0            0            
     Unknown               4     0.979          1            0            
     Unknown               5     1.000          1            0            
     Unknown               6     1.000          1            0            


Στο τερματικό του VS Code, από τον φάκελο `thesis_ids` (με ενεργό το `.venv`):

```
python src/predict_intrusion.py data/example_sessions.csv
```

Οι προβλέψεις γράφονται στο `data/example_sessions_predictions.csv`.